In [2]:
%pip install -q -U transformers
%pip install -q -U ultralytics

In [3]:
# Install/Reinstall torch and transformers for compatibility
# IMPORTANT: After running this cell, please restart the runtime (Runtime > Restart runtime...)
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

# Standard library
from dataclasses import dataclass
from pathlib import Path
from typing import Protocol, TypeAlias

# Numerical processing and model inference
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

# Image loading and result visualisation
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import display

# DINOv2 model and preprocessing
from transformers import AutoImageProcessor, AutoModel

In [ ]:
PROJECT_DIR = Path(
    "/content/drive/MyDrive/Project_AUTO_ReID"
)

REFERENCE_DIR = PROJECT_DIR / "references"
QUERY_DIR = PROJECT_DIR / "queries"
EMBEDDINGS_DIR = PROJECT_DIR / "embeddings"
EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = PROJECT_DIR / "results"


In [ ]:
TEMP_CROPPED_DIR = PROJECT_DIR / "temp_cropped_images"
TEMP_CROPPED_DIR.mkdir(parents=True, exist_ok=True)
print("Temporary cropped images folder:", TEMP_CROPPED_DIR)

Temporary cropped images folder: c:\Users\jaska\OneDrive\SourceRepository\Project_AUTO\notebooks\ReID_Tuning\temp_cropped_images


In [ ]:
print("Project:", PROJECT_DIR)
print("References found:", REFERENCE_DIR.exists())
print("Queries found:", QUERY_DIR.exists())
print("Embeddings folder:", EMBEDDINGS_DIR)
print("Results folder:", RESULTS_DIR)

Project: /content/c:\Users\jaska\OneDrive\SourceRepository\Project_AUTO\notebooks\ReID_Tuning
References found: False
Queries found: False
Embeddings folder: /content/c:\Users\jaska\OneDrive\SourceRepository\Project_AUTO\notebooks\ReID_Tuning/embeddings
Results folder: /content/c:\Users\jaska\OneDrive\SourceRepository\Project_AUTO\notebooks\ReID_Tuning/results


In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}

def get_image_paths(folder: Path) -> list[Path]:
    return sorted(
        path
        for path in folder.rglob("*")
        if path.is_file()
        and path.suffix.lower() in IMAGE_EXTENSIONS
    )

for folder in sorted(REFERENCE_DIR.iterdir()):
    if folder.is_dir():
        print(
            f"Reference — {folder.name}: "
            f"{len(get_image_paths(folder))} images"
        )

for folder in sorted(QUERY_DIR.iterdir()):
    if folder.is_dir():
        print(
            f"Query — {folder.name}: "
            f"{len(get_image_paths(folder))} images"
        )

FileNotFoundError: [Errno 2] No such file or directory: '/content/c:\\Users\\jaska\\OneDrive\\SourceRepository\\Project_AUTO\\notebooks\\ReID_Tuning/references'

In [ ]:
def _crop_image_with_bbox(
    original_image: Image.Image,
    bbox_xyxy: tuple[int, int, int, int],
    original_img_path: Path
) -> Path:
    x1, y1, x2, y2 = bbox_xyxy

    # Calculate width and height of the original bbox
    bbox_width = x2 - x1
    bbox_height = y2 - y1

    # Calculate 5% padding based on bbox dimensions
    padding_x = int(0.05 * bbox_width)
    padding_y = int(0.05 * bbox_height)

    # Apply padding, ensuring it stays within image bounds
    x1_padded = max(0, x1 - padding_x)
    y1_padded = max(0, y1 - padding_y)
    x2_padded = min(original_image.width, x2 + padding_x)
    y2_padded = min(original_image.height, y2 + padding_y)

    cropped_image = original_image.crop((x1_padded, y1_padded, x2_padded, y2_padded))

    # Save cropped image to a temporary path
    temp_filename = f"{original_img_path.stem}_yolo_padded_cropped{original_img_path.suffix}"
    temp_cropped_path = TEMP_CROPPED_DIR / temp_filename
    temp_cropped_path.parent.mkdir(parents=True, exist_ok=True)  # Ensure dir exists
    cropped_image.save(temp_cropped_path)
    return temp_cropped_path

def detect_object(img_path_list: list[Path]) -> list[tuple[Path, tuple[int, int, int, int] | None]]:
    processed_results = []
    for img_path in img_path_list:
        original_image = Image.open(img_path).convert("RGB")
        # Perform object detection
        # Using verbose=False to suppress print statements from YOLO
        results = yolo_model.predict(str(img_path), conf=0.25, iou=0.45, verbose=False)

        if results and results[0].boxes:  # Check if any objects were detected
            # Get the bounding boxes and areas
            boxes = results[0].boxes.xywh.cpu().numpy()  # xywh format
            areas = boxes[:, 2] * boxes[:, 3]  # width * height

            # Find the index of the largest bounding box
            largest_box_idx = np.argmax(areas)
            largest_box = boxes[largest_box_idx]

            # Convert xywh to xyxy for cropping (x_min, y_min, x_max, y_max)
            x_center, y_center, width, height = largest_box
            x1 = max(0, int(x_center - width / 2))
            y1 = max(0, int(y_center - height / 2))
            x2 = min(original_image.width, int(x_center + width / 2))
            y2 = min(original_image.height, int(y_center + height / 2))

            # Store the bbox relative to the original image
            bbox_original = (x1, y1, x2, y2)

            # Call the new cropping function to get the cropped image path
            cropped_path = _crop_image_with_bbox(original_image, bbox_original, img_path)
            processed_results.append((cropped_path, bbox_original))
        else:
            # If no objects detected, return original image path and no bbox
            processed_results.append((img_path, None))
    return processed_results

In [ ]:
MODEL_NAME = "facebook/dinov2-base"

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

processor = AutoImageProcessor.from_pretrained(MODEL_NAME)

model = AutoModel.from_pretrained(MODEL_NAME)
model = model.to(DEVICE)
model.eval()

print("Model:", MODEL_NAME)
print("Device:", DEVICE)

In [ ]:
from transformers import Sam2Processor, Sam2Model


SAM_MODEL_NAME = "facebook/sam2-hiera-tiny"

sam_processor = Sam2Processor.from_pretrained(
    SAM_MODEL_NAME
)

sam_model = Sam2Model.from_pretrained(
    SAM_MODEL_NAME
).to(DEVICE)

sam_model.eval()

print("Segmentation model:", SAM_MODEL_NAME)
print("Device:", DEVICE)

In [ ]:
from ultralytics import YOLO

YOLO_MODEL_NAME = "yolov8n.pt"  # Using yolov8n as a common lightweight model

yolo_model = YOLO(YOLO_MODEL_NAME)
yolo_model.to(DEVICE)  # Move YOLO model to the same device as DINOv2 and SAM
yolo_model.eval()  # Set to evaluation mode

print("Object detection model:", YOLO_MODEL_NAME)
print("Device:", DEVICE)

In [ ]:
def apply_sam_mask(
    image_to_mask_path: Path, # Changed from original_image_path
    object_point: tuple[int, int] | None = None,
    bbox: tuple[int, int, int, int] | None = None, # New argument for bbox prompt
    background_colour: tuple[int, int, int] = (128, 128, 128),
) -> Image.Image:
    """
    Pass an image through SAM 2, select the best non-empty mask,
    and replace every pixel where the mask equals 0 with grey.

    Mask value 1/True  -> retain the original image pixel
    Mask value 0/False -> replace with background_colour
    """

    # 1. Load the image. This is now expected to be the (potentially cropped) image.
    image = Image.open(image_to_mask_path).convert("RGB")
    width, height = image.size

    # SAM 2 requires a prompt identifying the target object.
    # Prioritize bounding box prompt over object point.
    input_points = None
    input_labels = None
    input_boxes = None

    if bbox is not None:
        # SAM expects boxes in format [x_min, y_min, x_max, y_max]
        # The bbox is now expected to be relative to `image_to_mask_path`
        input_boxes = [[[float(bbox[0]), float(bbox[1]), float(bbox[2]), float(bbox[3])]]]
    elif object_point is not None:
        point_x, point_y = object_point
        input_points = [[[[
                float(point_x),
                float(point_y),
            ]
        ]]]
        # Label 1 means that the point is on the target object.
        input_labels = [[[1]]]
    else:
        # Default to image center if no bbox or object_point provided
        object_point = (
            width // 2,
            height // 2,
        )
        point_x, point_y = object_point
        input_points = [[[[
                float(point_x),
                float(point_y),
            ]
        ]]]
        input_labels = [[[1]]]

    # 2. Process the image and prompt, then move tensors to GPU.
    inputs = sam_processor(
        images=image,
        input_points=input_points,
        input_labels=input_labels,
        input_boxes=input_boxes, # Pass input_boxes
        return_tensors="pt",
    ).to(DEVICE)

    # 3. Pass the image through SAM 2.
    with torch.inference_mode():
        outputs = sam_model(
            **inputs,
            multimask_output=True,
        )

    # 4. Resize SAM's masks to the original image resolution.
    processed_masks = sam_processor.post_process_masks(
        outputs.pred_masks.cpu(),
        inputs["original_sizes"].cpu(),
        mask_threshold=0.0,
        binarize=True,
    )[0]

    # processed_masks shape:
    # [number_of_objects, number_of_candidate_masks, height, width]

    mask_scores = (
        outputs.iou_scores[0, 0]
        .detach()
        .cpu()
    )

    # 5. Exclude any completely empty candidate masks.
    mask_areas = torch.tensor([
        processed_masks[0, index].sum().item()
        for index in range(processed_masks.shape[1])
    ])

    valid_masks = mask_areas > 0

    if not valid_masks.any():
        raise RuntimeError(
            "SAM 2 returned only empty masks. "
            "Choose an object_point located inside the object."
        )

    valid_scores = mask_scores.clone()
    valid_scores[~valid_masks] = float("-inf")

    best_mask_index = int(
        torch.argmax(valid_scores).item()
    )

    # 6. Extract the selected binary mask.
    foreground_mask = (
        processed_masks[0, best_mask_index]
        .numpy()
        .astype(bool)
    )

    # 7. Copy the original image.
    masked_image_np = np.array(
        image,
        dtype=np.uint8,
        copy=True,
    )

    # Wherever the mask is 0/False, replace the pixel with grey.
    masked_image_np[foreground_mask] = np.asarray(
        background_colour,
        dtype=np.uint8,
    )

    retained_percentage = (
        foreground_mask.sum()
        / foreground_mask.size
        * 100
    )

    print("Selected mask:", best_mask_index)
    print(
        "Mask score:",
        f"{mask_scores[best_mask_index].item():.4f}",
    )
    print(
        "Image retained:",
        f"{retained_percentage:.2f}%",
    )

    return Image.fromarray(masked_image_np)

### Visualizing Preprocessed Reference Images

This function demonstrates the YOLO cropping and SAM masking process on the first image of each identity within your `reference_gallery`. This helps to visualize how the images are preprocessed before their embeddings are created.

In [ ]:
import torch

@torch.inference_mode()
def create_embedding(image: Image.Image) -> torch.Tensor:
    # image: Image.Image is now directly passed, removing the need to open from path

    inputs = processor(
        images=image,
        return_tensors="pt",
    )

    inputs = {
        name: tensor.to(DEVICE)
        for name, tensor in inputs.items()
    }

    outputs = model(**inputs)

    # CLS token represents the overall image.
    embedding = outputs.last_hidden_state[:, 0, :]

    # Normalize so dot product can be used as cosine similarity.
    embedding = F.normalize(
        embedding,
        p=2,
        dim=1,
    )

    return embedding.squeeze(0).cpu()

In [ ]:
def build_reference_gallery():
    # Build locally so an interruption cannot erase a valid gallery.
    new_gallery = {}

    for identity_folder in sorted(REFERENCE_DIR.iterdir()):
        if not identity_folder.is_dir():
            continue

        original_paths = get_image_paths(identity_folder)
        if not original_paths:
            continue

        successful_paths = []
        embeddings_list = []

        for original_path in original_paths:
            processed_results = detect_object([original_path])
            if not processed_results:
                print(f"Skipped: no detection for {original_path}")
                continue

            cropped_path, _ = processed_results[0]
            cropped_image = Image.open(cropped_path).convert("RGB")
            cropped_width, cropped_height = cropped_image.size
            sam_bbox = (0, 0, cropped_width, cropped_height)

            masked_image = apply_sam_mask(
                image_to_mask_path=cropped_path,
                bbox=sam_bbox,
            )

            embeddings_list.append(create_embedding(masked_image))
            successful_paths.append(str(original_path))

        if not embeddings_list:
            print(f"Skipped identity: {identity_folder.name}")
            continue

        embeddings = torch.stack(embeddings_list)
        new_gallery[identity_folder.name] = {
            "paths": successful_paths,
            "embeddings": embeddings,
        }
        print(identity_folder.name, embeddings.shape)

    if not new_gallery:
        raise RuntimeError(
            "No reference embeddings were created. "
            "Check REFERENCE_DIR and object detections."
        )

    return new_gallery


# Assignment occurs only after the complete build succeeds.
reference_gallery = build_reference_gallery()
print("Gallery identities:", list(reference_gallery.keys()))

In [ ]:
GALLERY_PATH = EMBEDDINGS_DIR / "reference_gallery.pt"

torch.save(
    {
        "model_name": MODEL_NAME,
        "gallery": reference_gallery,
    },
    GALLERY_PATH,
)

print("Saved:", GALLERY_PATH)

In [ ]:
# def show_processed_reference_images():
#     for identity_name, evidence in reference_gallery.items():
#         print(f"\n--- Processing {identity_name} ---")
#         # Get the original path of the first image for this identity
#         original_path = Path(evidence["paths"][0])

#         # 1. Detect object and get cropped image path
#         # detect_object expects a list, even for a single image
#         processed_results = detect_object([original_path])
#         cropped_path, _ = processed_results[0]

#         # 2. Load the cropped image to determine its dimensions
#         cropped_image = Image.open(cropped_path).convert("RGB")
#         cropped_width, cropped_height = cropped_image.size

#         # 3. Define a bbox for SAM that covers the entire cropped image
#         sam_bbox_for_cropped_img = (0, 0, cropped_width, cropped_height)

#         # 4. Apply SAM mask to the cropped image
#         masked_image = apply_sam_mask(
#             image_to_mask_path=cropped_path,
#             bbox=sam_bbox_for_cropped_img
#         )

#         print(f"Original Image ({identity_name}): {original_path.name}")
#         display(Image.open(original_path).convert("RGB"))

#         print(f"Cropped Image ({identity_name}):")
#         display(cropped_image)

#         print(f"Masked Image ({identity_name}):")
#         display(masked_image)

# # Call the function to display the processed images
# show_processed_reference_images()

TODO: implement YOLO detection and then passoff to SAM before segmentation

In [ ]:
def compare_query(query_embedding: torch.Tensor):
    gallery = globals().get("reference_gallery")

    if not gallery:
        raise RuntimeError(
            "reference_gallery is empty or undefined. "
            "Run the complete gallery-building cell first."
        )

    query_embedding = query_embedding.squeeze().cpu()
    comparison_rows = []
    identity_rows = []

    for candidate_identity, evidence in gallery.items():
        reference_embeddings = evidence["embeddings"].cpu()
        reference_paths = evidence["paths"]

        if reference_embeddings.ndim != 2:
            raise ValueError(
                f"{candidate_identity} embeddings must have shape "
                f"[number_of_images, embedding_size], but received "
                f"{tuple(reference_embeddings.shape)}."
            )

        if reference_embeddings.shape[1] != query_embedding.shape[0]:
            raise ValueError(
                f"Embedding dimension mismatch for {candidate_identity}: "
                f"gallery={reference_embeddings.shape[1]}, "
                f"query={query_embedding.shape[0]}. "
                f"Regenerate the gallery using {MODEL_NAME}."
            )

        if len(reference_paths) != len(reference_embeddings):
            raise ValueError(
                f"Path/embedding count mismatch for {candidate_identity}: "
                f"{len(reference_paths)} paths and "
                f"{len(reference_embeddings)} embeddings."
            )

        # Normalized embeddings: dot product equals cosine similarity.
        similarities = reference_embeddings @ query_embedding

        # Diagnostic only: summarize this identity using its top-k matches.
        k = 3
        effective_k = min(k, len(similarities))
        top_k_similarities = torch.topk(
            similarities,
            k=effective_k,
        ).values
        top_k_mean = top_k_similarities.mean()

        # Keep the strongest individual reference as display evidence.
        best_index = int(similarities.argmax().item())
        best_reference_path = Path(reference_paths[best_index])
        identity_rows.append({
            "candidate_identity": candidate_identity,
            "reference_image": best_reference_path.name,
            "reference_path": str(best_reference_path),
            "best_similarity": float(similarities[best_index].item()),
            "top_k_mean": float(top_k_mean.item()),
        })

        print(
            candidate_identity,
            "all similarities:", similarities.tolist(),
            "top-k mean:", top_k_mean.item(),
        )

        for index, similarity in enumerate(similarities):
            reference_path = Path(reference_paths[index])
            comparison_rows.append({
                "query_identity": "UNKNOWN",
                "query_image": "UNKNOWN",
                "candidate_identity": candidate_identity,
                "reference_image": reference_path.name,
                "reference_path": str(reference_path),
                "best_similarity": float(similarity.item()),
            })

    if not comparison_rows:
        raise RuntimeError(
            "The gallery exists, but no comparisons were generated."
        )

    all_comparisons = (
        pd.DataFrame(comparison_rows)
        .sort_values("best_similarity", ascending=False)
        .reset_index(drop=True)
    )

    identity_matches = (
        pd.DataFrame(identity_rows)
        .sort_values("top_k_mean", ascending=False)
        .reset_index(drop=True)
    )

    return all_comparisons, identity_matches

The `reference_gallery` needs to be re-generated using the current DINOv2-base model to match the embedding dimensions of the query images. This will resolve the `RuntimeError: size mismatch`.

Now that the `reference_gallery` has been updated with the correct embedding dimensions, we can proceed with comparing query images and generating the summary.

In [ ]:
MATCH_VISUALS_DIR = RESULTS_DIR / "match_visuals"
MATCH_VISUALS_DIR.mkdir(parents=True, exist_ok=True)

def show_query_match(query_path: Path):
    # 1. Preprocess the query image: Crop and Mask
    # detect_object expects a list
    processed_query_results = detect_object([query_path])
    cropped_query_path, _ = processed_query_results[0]

    # Load the cropped image to determine its dimensions for SAM bbox
    cropped_query_image = Image.open(cropped_query_path).convert("RGB")
    cropped_width, cropped_height = cropped_query_image.size
    sam_bbox_for_cropped_query = (0, 0, cropped_width, cropped_height)

    # Apply SAM mask to the cropped query image
    masked_query_image = apply_sam_mask(
        image_to_mask_path=cropped_query_path,
        bbox=sam_bbox_for_cropped_query
    )

    # Create embedding from the masked query image
    query_embedding = create_embedding(masked_query_image)

    # Now pass the pre-computed query_embedding to compare_query
    all_comparisons, identity_matches = compare_query(query_embedding)

    best_match = identity_matches.iloc[0]
    best_reference_path = Path(
        best_match["reference_path"]
    )

    figure, axes = plt.subplots(
        1,
        2,
        figsize=(10, 4),
    )

    axes[0].imshow(
        Image.open(query_path).convert("RGB") # Display original query image
    )
    axes[0].set_title(
        f"Query\n"
        f"{query_path.parent.name}/{query_path.name}"
    )
    axes[0].axis("off")

    axes[1].imshow(
        Image.open(best_reference_path).convert("RGB") # Display original best match image
    )
    axes[1].set_title(
        f"Best match: {best_match['candidate_identity']}\n"
        f"Best reference: {best_match['reference_image']}\n"
        f"Top-k mean: {best_match['top_k_mean']:.4f}\n"
        f"Best similarity: {best_match['best_similarity']:.4f}"
    )
    axes[1].axis("off")

    figure.tight_layout()

    output_name = (
        f"{query_path.parent.name}_"
        f"{query_path.stem}_match.png"
    )

    output_path = MATCH_VISUALS_DIR / output_name

    figure.savefig(
        output_path,
        dpi=150,
        bbox_inches="tight",
    )

    plt.show()
    plt.close(figure)

    print("Visual saved:", output_path)

    display(
        identity_matches[
            [
                "candidate_identity",
                "reference_image",
                "top_k_mean",
                "best_similarity",
            ]
        ].style.format({
            "top_k_mean": "{:.4f}",
            "best_similarity": "{:.4f}"
        })
    )

    return all_comparisons, identity_matches

In [ ]:
all_pairwise_results = []
query_summary_rows = []

for query_path in get_image_paths(QUERY_DIR):
    comparisons, identity_matches = show_query_match(query_path)
    all_pairwise_results.append(comparisons)

    if len(identity_matches) < 2:
        raise RuntimeError(
            "At least two reference identities are required "
            "to calculate the similarity margin."
        )

    best = identity_matches.iloc[0]
    second = identity_matches.iloc[1]
    expected_identity = query_path.parent.name

    query_summary_rows.append({
        "query_identity": expected_identity,
        "query_image": query_path.name,
        "predicted_identity": best["candidate_identity"],
        "matched_reference": best["reference_image"],
        "best_similarity": best["best_similarity"],
        "top_k_mean": best["top_k_mean"],
        "second_identity": second["candidate_identity"],
        "second_similarity": second["best_similarity"],
        "second_top_k_mean": second["top_k_mean"],
        "margin": best["top_k_mean"] - second["top_k_mean"],
        "correct": (
            None
            if expected_identity == "unknown"
            else best["candidate_identity"] == expected_identity
        ),
    })

if not all_pairwise_results:
    raise RuntimeError("No query images were found in QUERY_DIR.")

pairwise_results = pd.concat(all_pairwise_results, ignore_index=True)
query_summary = pd.DataFrame(query_summary_rows)

display(
    query_summary.style.format({
        "best_similarity": "{:.4f}",
        "top_k_mean": "{:.4f}",
        "second_similarity": "{:.4f}",
        "second_top_k_mean": "{:.4f}",
        "margin": "{:.4f}",
    })
)

In [ ]:
pairwise_results.to_csv(
    RESULTS_DIR / "all_pairwise_similarities.csv",
    index=False,
)

query_summary.to_csv(
    RESULTS_DIR / "query_match_summary.csv",
    index=False,
)

print("Results saved successfully.")